# Modélisation GMM des textures MEB

Modéliser chaque texture par un GMM dans l'espace `block_0`.  
Nombre optimal de composantes par BIC, qualité du modèle, détection d'outliers, visualisation des modes.

In [ ]:
import h5py
import numpy as np
import json
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import confusion_matrix
from tqdm.auto import tqdm

ROOT     = Path('..').resolve()
DB_PATH  = ROOT / 'data' / 'feature_database' / 'database_meb.h5'
CFG_PATH = ROOT / 'PatchTagger_Output' / 'config' / 'config.json'
IMG_DIR  = ROOT / 'Image_Ouassim'
SEED         = 42
PCA_DIM      = 50
N_COMP_RANGE = [1, 2, 3, 4, 5]
CATS_EXCLUDE = [2, 8, 10, 11, 12, 13]
KEY          = 'block_0'

with open(CFG_PATH) as _f:
    _gmm_cfg = json.load(_f)
CATEGORIES = {int(k): v['name']
              for k, v in _gmm_cfg['available_categories'].items()}

with h5py.File(DB_PATH, 'r') as _h5:
    IMAGE_NAMES  = _h5['metadata/image_names'][:]
    POSITIONS    = _h5['metadata/positions'][:]
    CATEGORY_IDS = _h5['metadata/category_ids'][:]

CATS_VALID = sorted([
    c for c in np.unique(CATEGORY_IDS)
    if c not in CATS_EXCLUDE and (CATEGORY_IDS == c).sum() >= 30
])

# PCA-50d global — fit sur patches valides
_gmm_mask_valid = np.isin(CATEGORY_IDS, CATS_VALID)
with h5py.File(DB_PATH, 'r') as _h5:
    X_all = _h5['features'][KEY][:]
_gmm_pca = PCA(n_components=PCA_DIM, random_state=SEED)
_gmm_pca.fit(X_all[_gmm_mask_valid])

def get_cat_features(_gmm_c):
    """Features PCA + L2-norm pour une catégorie."""
    _gmm_idx = np.where(CATEGORY_IDS == _gmm_c)[0]
    _gmm_X   = _gmm_pca.transform(X_all[_gmm_idx])
    _gmm_n   = np.linalg.norm(_gmm_X, axis=1, keepdims=True)
    _gmm_X   = _gmm_X / np.where(_gmm_n < 1e-8, 1.0, _gmm_n)
    return _gmm_X, _gmm_idx

print(f'ROOT    : {ROOT}')
print(f'DB      : {DB_PATH}')
print(f'Catégories valides ({len(CATS_VALID)}) : {CATS_VALID}')
print(f'Patches totaux : {len(CATEGORY_IDS)}')

## Cellule 1 — Nombre optimal de composantes (BIC)

In [ ]:
gmm_models = {}

print(f'{"Catégorie":<25} │ {"N opt":>5} │ {"BIC":>10} │ {"log-lik":>10}')
print('─' * 60)

for _gmm_c in tqdm(CATS_VALID, desc='GMM par catégorie'):
    _gmm_X_c, _gmm_idx_c = get_cat_features(_gmm_c)

    _gmm_bics   = {}
    _gmm_aics   = {}
    _gmm_models = {}

    for _gmm_n in N_COMP_RANGE:
        if len(_gmm_X_c) < _gmm_n * 5:
            continue
        _gmm_gm = GaussianMixture(
            n_components    = _gmm_n,
            covariance_type = 'full',
            random_state    = SEED,
            max_iter        = 200,
            n_init          = 3,
        )
        _gmm_gm.fit(_gmm_X_c)
        _gmm_bics[_gmm_n]   = _gmm_gm.bic(_gmm_X_c)
        _gmm_aics[_gmm_n]   = _gmm_gm.aic(_gmm_X_c)
        _gmm_models[_gmm_n] = _gmm_gm

    _gmm_n_opt = min(_gmm_bics, key=_gmm_bics.get)

    gmm_models[_gmm_c] = {
        'gmm'    : _gmm_models[_gmm_n_opt],
        'n_opt'  : _gmm_n_opt,
        'bic'    : _gmm_bics,
        'aic'    : _gmm_aics,
        'loglik' : _gmm_models[_gmm_n_opt].score(_gmm_X_c),
        'X_c'    : _gmm_X_c,
        'idx_c'  : _gmm_idx_c,
    }

    print(f'{CATEGORIES[_gmm_c]:<25} │ {_gmm_n_opt:>5} │ '
          f'{_gmm_bics[_gmm_n_opt]:>10.1f} │ '
          f'{_gmm_models[_gmm_n_opt].score(_gmm_X_c):>10.3f}')

## Cellule 2 — Courbes BIC par catégorie

In [ ]:
_gmm_fig, _gmm_ax = plt.subplots(figsize=(12, 6))

for _gmm_c in CATS_VALID:
    _gmm_bics = gmm_models[_gmm_c]['bic']
    _gmm_ns   = sorted(_gmm_bics.keys())
    _gmm_vals = np.array([_gmm_bics[n] for n in _gmm_ns], dtype=float)
    # Min-max normalisation pour comparaison visuelle
    _gmm_vmin = _gmm_vals.min()
    _gmm_vmax = _gmm_vals.max()
    _gmm_vals_norm = (_gmm_vals - _gmm_vmin) / (_gmm_vmax - _gmm_vmin + 1e-10)
    _gmm_color = _gmm_cfg['available_categories'][str(_gmm_c)]['color']
    _gmm_ax.plot(
        _gmm_ns, _gmm_vals_norm, 'o-', lw=1.8, ms=6,
        color=_gmm_color,
        label=f'{CATEGORIES[_gmm_c]} (opt={gmm_models[_gmm_c]["n_opt"]})'
    )

_gmm_ax.set_xlabel('Nombre de composantes GMM', fontsize=11)
_gmm_ax.set_ylabel('BIC normalisé (min = meilleur)', fontsize=11)
_gmm_ax.set_title(
    'Sélection du nombre de composantes par BIC\n'
    f'{KEY} · minimum = nombre optimal de modes',
    fontsize=12
)
_gmm_ax.set_xticks(N_COMP_RANGE)
_gmm_ax.legend(fontsize=8)
_gmm_ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('gmm_bic_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : gmm_bic_selection.png')

## Cellule 3 — Matrice de confusion probabiliste

In [ ]:
_gmm_y_true = []
_gmm_y_pred = []

for _gmm_c_true in tqdm(CATS_VALID, desc='Confusion GMM'):
    _gmm_X_c, _ = get_cat_features(_gmm_c_true)

    # Log-vraisemblance sous chaque GMM
    _gmm_logliks = np.zeros((len(_gmm_X_c), len(CATS_VALID)))
    for _gmm_j, _gmm_c_model in enumerate(CATS_VALID):
        _gmm_logliks[:, _gmm_j] = gmm_models[_gmm_c_model]['gmm'].score_samples(_gmm_X_c)

    _gmm_pred_idx = _gmm_logliks.argmax(axis=1)
    _gmm_preds    = [CATS_VALID[k] for k in _gmm_pred_idx]

    _gmm_y_true.extend([_gmm_c_true] * len(_gmm_X_c))
    _gmm_y_pred.extend(_gmm_preds)

cm = confusion_matrix(_gmm_y_true, _gmm_y_pred,
                      labels=CATS_VALID, normalize='true') * 100

_gmm_fig, _gmm_ax = plt.subplots(figsize=(9, 8))
_gmm_im     = _gmm_ax.imshow(cm, cmap='Blues', vmin=0, vmax=100)
_gmm_labels = [CATEGORIES[c] for c in CATS_VALID]
_gmm_ax.set_xticks(range(len(CATS_VALID)))
_gmm_ax.set_yticks(range(len(CATS_VALID)))
_gmm_ax.set_xticklabels(_gmm_labels, rotation=45, ha='right', fontsize=9)
_gmm_ax.set_yticklabels(_gmm_labels, fontsize=9)
_gmm_ax.set_xlabel('Texture prédite (max vraisemblance GMM)', fontsize=10)
_gmm_ax.set_ylabel('Texture réelle', fontsize=10)

for _gmm_i in range(len(CATS_VALID)):
    for _gmm_j in range(len(CATS_VALID)):
        _gmm_val = cm[_gmm_i, _gmm_j]
        _gmm_ax.text(
            _gmm_j, _gmm_i, f'{_gmm_val:.0f}',
            ha='center', va='center',
            color='white' if _gmm_val > 50 else '#1a1a2e',
            fontsize=8,
            fontweight='bold' if _gmm_i == _gmm_j else 'normal'
        )

plt.colorbar(_gmm_im, ax=_gmm_ax, fraction=0.046, pad=0.04)
_gmm_ax.set_title(
    'Confusion probabiliste GMM — block_0\n'
    'Classification par max vraisemblance',
    fontsize=12
)
plt.tight_layout()
plt.savefig('gmm_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : gmm_confusion.png')

## Cellule 4 — Détection d'outliers

In [ ]:
print('=== Outliers par catégorie (5% plus faible vraisemblance) ===\n')

outliers = {}
for _gmm_c in CATS_VALID:
    _gmm_X_c   = gmm_models[_gmm_c]['X_c']
    _gmm_idx_c = gmm_models[_gmm_c]['idx_c']
    _gmm_gm    = gmm_models[_gmm_c]['gmm']

    _gmm_logliks   = _gmm_gm.score_samples(_gmm_X_c)
    _gmm_threshold = np.percentile(_gmm_logliks, 5)
    _gmm_mask_out  = _gmm_logliks < _gmm_threshold

    outliers[_gmm_c] = {
        'idx'   : _gmm_idx_c[_gmm_mask_out],
        'loglik': _gmm_logliks[_gmm_mask_out],
    }
    print(f'{CATEGORIES[_gmm_c]:<25} : {_gmm_mask_out.sum()} outliers '
          f'(loglik < {_gmm_threshold:.1f})')

## Cellule 5 — Visualiser composantes GMM (LDA 2D)

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# LDA global — projection en 2D
_gmm_X_lda_all = []
_gmm_y_lda_all = []
for _gmm_c in CATS_VALID:
    _gmm_X_c, _ = get_cat_features(_gmm_c)
    _gmm_X_lda_all.append(_gmm_X_c)
    _gmm_y_lda_all.extend([_gmm_c] * len(_gmm_X_c))
_gmm_X_lda_all = np.vstack(_gmm_X_lda_all)
_gmm_y_lda_all = np.array(_gmm_y_lda_all)

_gmm_lda = LinearDiscriminantAnalysis(n_components=2)
_gmm_X_2d = _gmm_lda.fit_transform(_gmm_X_lda_all, _gmm_y_lda_all)

_gmm_fig, _gmm_ax = plt.subplots(figsize=(12, 9))

for _gmm_c in CATS_VALID:
    _gmm_mask  = _gmm_y_lda_all == _gmm_c
    _gmm_color = _gmm_cfg['available_categories'][str(_gmm_c)]['color']
    _gmm_ax.scatter(
        _gmm_X_2d[_gmm_mask, 0], _gmm_X_2d[_gmm_mask, 1],
        c=_gmm_color, s=10, alpha=0.4,
        label=f'{CATEGORIES[_gmm_c]} ({gmm_models[_gmm_c]["n_opt"]} modes)'
    )

    # Projeter les moyennes GMM dans l'espace LDA
    _gmm_gm        = gmm_models[_gmm_c]['gmm']
    _gmm_means_lda = _gmm_lda.transform(_gmm_gm.means_)
    _gmm_ax.scatter(
        _gmm_means_lda[:, 0], _gmm_means_lda[:, 1],
        c=_gmm_color, s=300, marker='*',
        edgecolors='black', linewidths=1.5, zorder=5
    )

_gmm_ax.set_xlabel('LDA 1', fontsize=11)
_gmm_ax.set_ylabel('LDA 2', fontsize=11)
_gmm_ax.set_title(
    f'Modèles GMM par texture — {KEY}\n'
    '★ = composantes gaussiennes · points = patches',
    fontsize=12
)
_gmm_ax.legend(fontsize=8, markerscale=1.5)
_gmm_ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('gmm_components_lda.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : gmm_components_lda.png')

## Cellule 6 — Résumé modélisation + sauvegarde

In [ ]:
import pickle

print('=== MODÈLE DE TEXTURE GMM — RÉSUMÉ ===\n')
print(f'{"Texture":<25} │ {"Modes":>5} │ {"log-lik":>9} │ {"Recall GMM":>10}')
print('─' * 60)

for _gmm_i, _gmm_c in enumerate(CATS_VALID):
    _gmm_recall = cm[_gmm_i, _gmm_i]
    print(f'{CATEGORIES[_gmm_c]:<25} │ '
          f'{gmm_models[_gmm_c]["n_opt"]:>5} │ '
          f'{gmm_models[_gmm_c]["loglik"]:>9.2f} │ '
          f'{_gmm_recall:>9.1f}%')

_gmm_save_path = ROOT / 'outputs' / 'gmm_texture_models.pkl'
_gmm_save_path.parent.mkdir(parents=True, exist_ok=True)
with open(_gmm_save_path, 'wb') as _f:
    pickle.dump(
        {c: {'gmm': gmm_models[c]['gmm'], 'n_opt': gmm_models[c]['n_opt']}
         for c in CATS_VALID},
        _f
    )
print(f'\nModèles sauvegardés → {_gmm_save_path}')